# Лабораторная работа №4. Задача регрессии

In [ ]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.ensemble import BaggingRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearnex.ensemble import RandomForestRegressor
from matplotlib import pyplot as plt
import seaborn as sns


RANDOM_STATE = 13

## Загрузка данных и разделение выборки

In [ ]:
data = pd.read_csv("datasets/air.csv", index_col="Unnamed: 0")
data

Разделим выборку в формате 80 - 10 - 10

In [ ]:
X = data.drop("CO(GT)", axis=1)
y = data["CO(GT)"]

X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test =  train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=RANDOM_STATE)

print(f"Размер тренировочной выборки: {len(X_train)}\n"
      f"Размер валидационной выборки: {len(X_val)}\n"
      f"Размер тестовой выборки: {len(X_test)}")

## Обучение моделей

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, GridSearchCV

def train_models_with_cross_val(X_train, y_train, models_config):

    result={}

    for name, model_info in models_config.items():

        param_grid = model_info["param_grid"]
        model_obj = model_info["model"]

        grid_search_model = GridSearchCV(
            param_grid=param_grid,
            estimator= model_obj(),
            scoring="r2",
            cv=5,
            n_jobs=8
        )

        grid_search_model.fit(X_train, y_train)

        random_search_model = RandomizedSearchCV(
            param_distributions=param_grid,
            estimator=model_obj(),
            scoring="r2",
            random_state=RANDOM_STATE,
            cv=5,
            n_jobs=8
        )

        random_search_model.fit(X_train, y_train)

        def objective(trial):
            trial_params = {}
            for p_name, p_values in param_grid.items():
                if isinstance(p_values[0], float):
                    trial_params[p_name] = trial.suggest_float(p_name, min(p_values), max(p_values))
                else:
                    trial_params[p_name] = trial.suggest_categorical(p_name, p_values)

            current_model = model_obj(**trial_params,random_state=RANDOM_STATE)

            score = cross_val_score(current_model, X_train, y_train, cv=5, scoring='r2').mean()
            return score

        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=15, show_progress_bar=False)

        final_optuna_model = model_obj(**study.best_params,random_state=RANDOM_STATE)
        final_optuna_model.fit(X_train, y_train)

        result[name] = {}
        result[name]["GridSearch"] = {
            "model": grid_search_model.best_estimator_,
            "best_params": grid_search_model.best_params_
        }
        result[name]["RandomSearch"] = {
            "model": random_search_model.best_estimator_,
            "best_params": random_search_model.best_params_
        }
        result[name]["Optuna"] = {
            "model": final_optuna_model,
            "best_params": study.best_params
        }

    return result

In [ ]:
metrics = ["R2", "MSE", "RMSE", "MAE", "MAPE"]

columns = pd.MultiIndex.from_product([
    ["Train Data", "Test Data"],
    metrics
])

train_and_test_comparing = pd.DataFrame(columns=columns)
train_and_test_comparing

### Деревья решений

In [ ]:
dt_config = {
    "DT": {
        "model": DecisionTreeRegressor,
        "param_grid": {
            "criterion": ["squared_error", "friedman_mse", "absolute_error"],
            "max_depth": [None, 3, 5, 10, 15, 20, 30],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "max_features": [None, "sqrt", "log2"],
            "ccp_alpha": np.logspace(-4, -1, 10).tolist()
        }
    }
}

In [ ]:
tree_models = train_models_with_cross_val(X_train, y_train, dt_config)

In [ ]:
import pickle
with open("tree_models.pkl", "wb") as f:
    pickle.dump(tree_models, f)

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, \
    root_mean_squared_error


def get_scores(y_true, y_pred):
        r2 = round(r2_score(y_true, y_pred), 2)
        mse = round(mean_squared_error(y_true, y_pred), 4)
        rmse = round(root_mean_squared_error(y_true, y_pred), 4)
        mae = round(mean_absolute_error(y_true, y_pred), 4)
        mape = round(mean_absolute_percentage_error(y_true, y_pred), 4)
        return [r2, mse, rmse, mae, mape]

def input_into_table(Xtrain, Xtest, ytrain, ytest, model, name_of_model):
    train_pred = model.predict(Xtrain)
    test_pred = model.predict(Xtest)

    train_and_test_comparing.loc[name_of_model] = (get_scores(ytrain, train_pred)
                                                   + get_scores(ytest, test_pred))

In [ ]:
tree_models.items()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, tree_models["DT"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"DecisionTreeRegressor ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("DecisionTreeRegressor")

In [ ]:
train_and_test_comparing

### Бэггинг-регрессор

Для бэггинга применим лучшую модель, полученную в результате 2 лабораторной работы - полиномиальная регрессия 3 степени с Elastic-Net регуляризацией.

In [ ]:
from sklearn.linear_model import ElasticNet

pipeline = Pipeline([("poly", PolynomialFeatures(degree=3)),
                     ("model", ElasticNet(alpha=0.001, l1_ratio=0.1,))])

bag_reg = BaggingRegressor(estimator=pipeline, random_state=RANDOM_STATE)

bag_reg.fit(X_train, y_train)
input_into_table(X_train, X_test, y_train, y_test, bag_reg, "Bagging regressor")

In [ ]:
train_and_test_comparing

Применение бэггинга пока что показало лучшие результаты: метрика R2 равна 0.85.

In [ ]:
predict = bag_reg.predict(X_test)
sns.kdeplot(y_test, label="Истинные значения")
sns.kdeplot(predict, label="Предсказанные значения", linestyle="--")
plt.title("Bagging regressor")

In [ ]:
import pickle
with open("bag_model.pkl", "wb") as f:
    pickle.dump(bag_reg, f)

### Случайный лес

In [ ]:
forest_config = {
    "Random Forest":{
        "model": RandomForestRegressor,
        "param_grid":{
                'n_estimators': [100, 200, 300],
                'max_depth': [None, 10, 20, 30],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'max_features': ['sqrt', 'log2', None],
        }
    }

}

In [ ]:
random_forest_models = train_models_with_cross_val(X_train, y_train, forest_config)

In [ ]:
import pickle
with open("random_forest_models.pkl", "wb") as f:
    pickle.dump(random_forest_models, f)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, random_forest_models["Random Forest"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"Random Forest Regressor ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("RandomForestRegressor")

In [ ]:
train_and_test_comparing

Модель случайного леса показала еще лучшие результаты, чем предыдущие модели. Значение R2 возросло до 0.91.

### GradientBoostingRegressor


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_config = {
    "Gradient Boosting": {
        "model": GradientBoostingRegressor,
        "param_grid": {
            'n_estimators': [100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 8],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': ['sqrt', 'log2', None],
        }
    }
}

In [ ]:
gb_models = train_models_with_cross_val(X_train, y_train, gb_config)

In [ ]:
import pickle
with open("gradient_models.pkl", "wb") as f:
    pickle.dump(gb_models, f)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, gb_models["Gradient Boosting"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"GradientBoostingRegressor ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("GradientBoostingRegressor")

In [ ]:
train_and_test_comparing

Градиентный бустинг показал лучшие результаты на данный момент - на тренировочной выборке R2 равен 0.97.

### AdaBoostRegressor

In [ ]:
ab_config = {
    "AdaBoost": {
        "model": AdaBoostRegressor,
        "param_grid": {
            'n_estimators': [50, 100, 200, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0],
            'loss': ['linear', 'square', 'exponential']
        }
    }
}

In [ ]:
ab_models = train_models_with_cross_val(X_train, y_train, ab_config)

In [ ]:
import pickle
with open("adaboost_models.pkl", "wb") as f:
    pickle.dump(ab_models, f)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, ab_models["AdaBoost"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"AdaBoost ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("AdaBoost")

По графикам видно, что модель AdaBoost обобщает данные хуже предыдущих моделей.

In [ ]:
train_and_test_comparing

### StackingRegressor

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=20,
                           min_samples_split=2, min_samples_leaf=1, max_features=None)

stack_reg = StackingRegressor(estimators=[("poly", pipeline), ("forest", rf)])

stack_reg.fit(X_train, y_train)

In [ ]:
input_into_table(X_train, X_test, y_train, y_test, stack_reg, "Stacking Regressor")
train_and_test_comparing

Стэкинг тоже отработал неплохо, хотя хуже градиентного бустинга и случайного леса.

In [ ]:
import pickle
with open("stack.pkl", "wb") as f:
    pickle.dump(stack_reg, f)

In [ ]:
predict = stack_reg.predict(X_test)
sns.kdeplot(y_test, label="Истинные значения")
sns.kdeplot(predict, label="Предсказанные значения", linestyle="--")
plt.title("Stacking regressor")

### CatBoost

In [ ]:
cb_config = {
    "CatBoost": {
        "model": CatBoostRegressor,
        "param_grid": {
                'iterations': [100, 200, 300, 400, 500],
                'learning_rate': [0.01, 0.05, 0.1],
                'depth': [4, 6, 8],
                'l2_leaf_reg': [1, 3, 5, 7, 9],
        }
    }
}

In [ ]:
cb_models = train_models_with_cross_val(X_train, y_train, cb_config)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, cb_models["CatBoost"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"CatBoost ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("CatBoost")

In [ ]:
train_and_test_comparing

CatBoost также показал хорошие результаты - практически такие же хорошие, как и случайный лес.

### XGBRegressor

In [ ]:
from xgboost import XGBRegressor

xgb_config = {
    "XGBoost": {
        "model": XGBRegressor,
        "param_grid": {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [4, 6, 8],
            'reg_lambda': [1, 3, 5, 7, 9],
            'tree_method': ['hist'],
        }
    }
}

In [ ]:
xgb_models = train_models_with_cross_val(X_train, y_train, xgb_config)

In [ ]:
import pickle

with open("xgb.pkl", "wb") as f:
    pickle.dump(xgb_models, f)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, xgb_models["XGBoost"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"XGBoost ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("XGBoost")

In [ ]:
train_and_test_comparing

XGBoost-регрессор тоже показал хорошие результаты - коэффициент R2 равен 0.9.

### LGBMRegressor

In [ ]:
from lightgbm import LGBMRegressor

lgbm_config = {
    "LightGBM": {
        "model": LGBMRegressor,
        "param_grid": {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [4, 6, 8],
            'num_leaves': [15, 30],
        }
    }
}

In [ ]:
lgbm_models = train_models_with_cross_val(X_train, y_train, lgbm_config)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 10))
for ax, (name_of_method, model_info) in zip(axes, lgbm_models["LightGBM"].items()):
    model = model_info["model"]
    predict = model.predict(X_test)
    name_of_model = f"LightGBM ({name_of_method})"

    input_into_table(X_train, X_test, y_train, y_test, model, name_of_model)

    sns.kdeplot(y_test, label="Истинные значения", ax=ax)
    sns.kdeplot(predict, label="Предсказанные значения", ax=ax, linestyle="--")
    ax.set_title(name_of_method)
    ax.legend()

plt.suptitle("LightGBM")

In [ ]:
train_and_test_comparing

LightGBM также показывает хорошие результаты.

In [ ]:
import pickle
with open("light.pkl", "wb") as f:
    pickle.dump(lgbm_models, f)

### SketchBoost

In [ ]:
from py_boost import SketchBoost
from sklearn.model_selection import KFold

def objective(trial):
    params = {
        'loss': 'l2',
        'lr': trial.suggest_float('lr', 0.005, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'ntrees': trial.suggest_int('ntrees', 100, 1000, step=50),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 1.0, log=True)
    }


    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_t, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_t, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = SketchBoost(**params)
        model.fit(X_t, y_t)
        preds = model.predict(X_v)
        cv_scores.append(r2_score(y_v, preds))

    return np.mean(cv_scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print(f"Лучший R2: {study.best_value:.4f}")
print(f"Лучшие параметры: {study.best_params}")

final_sb_model = SketchBoost(loss='l2', **study.best_params)
final_sb_model.fit(X_train, y_train)

In [ ]:
import pickle

with open("sb_model.pkl", "rb") as f:
    final_sb_model = pickle.load(f)

In [ ]:
input_into_table(X_train, X_test, y_train, y_test, final_sb_model, "SketchBoost")
train_and_test_comparing

In [ ]:
import pickle

with open("sb_model.pkl", "rb") as f:
    final_sb_model = pickle.load(f)

In [ ]:
predict = final_sb_model.predict(X_test)


sns.kdeplot(y_test, label="Истинные значения")
sns.kdeplot(predict, label="Предсказанные значения", linestyle="--", color="red")

plt.legend()

plt.title("SketchBoost")

### Pycaret

In [ ]:
from pycaret.regression import *

s = setup(data = data, target = "CO(GT)", session_id=RANDOM_STATE)

In [ ]:
best_model = compare_models(n_select=5)

## Визуализации деревьев

In [ ]:
import pickle
with open("tree_models.pkl", "rb") as f:
    cb_models = pickle.load(f)

In [ ]:
from sklearn import tree
best_tree = tree_models["DT"]["GridSearch"]["model"]

plt.figure(figsize=(20, 10), dpi=300)

tree.plot_tree(best_tree,
               feature_names=X_train.columns,
               filled=True,
               max_depth=3,
               precision=2,
               rounded=True,
               proportion=True)

plt.title("Desicion Tree")

In [ ]:
import dtreeviz

viz_model = dtreeviz.model(best_tree,
                           X_train=X, y_train=y,
                           feature_names=X.columns,)

viz_model.view(depth_range_to_display=(0, 3))

In [ ]:
print(tree.export_text(best_tree, feature_names = X_train.columns))

In [ ]:
important_features_df = pd.Series(
    best_tree.feature_importances_, index=X_train.columns
).sort_values(ascending=True)

ax = important_features_df.plot.barh(legend=False, color='skyblue')

ax.set_title("Decision tree features importances")

In [ ]:
rf_best_model = random_forest_models["Random Forest"]["Optuna"]["model"]
rf_important_features_df = pd.Series(
    rf_best_model.feature_importances_, index=X_train.columns
).sort_values(ascending=True)

ax = rf_important_features_df.plot.barh(legend=False, color='skyblue')

ax.set_title("Random Forest features importances")

In [ ]:
sb_important_features_df = pd.Series(
    final_sb_model.get_feature_importance(), index=X_train.columns
).sort_values(ascending=True)

ax = sb_important_features_df.plot.barh(legend=False, color='skyblue')

ax.set_title("Sketch Boost features importances")

## Вывод

In [ ]:
head_of_table = []

for val in ["hold-out", "KFold"]:
    for metric in metrics:
        head_of_table.append(val + " " + metric)

hold_kross_table = pd.DataFrame(columns=head_of_table)
hold_kross_table

In [ ]:
import pickle

with open("sb_model.pkl", "rb") as f:
    final_sb_model = pickle.load(f)

In [ ]:
best_models = [best_tree,
               random_forest_models["Random Forest"]["Optuna"]["model"],
               ab_models["AdaBoost"]["Optuna"]["model"],
               gb_models["Gradient Boosting"]["GridSearch"]["model"],
               cb_models["CatBoost"]["GridSearch"]["model"],
               xgb_models["XGBoost"]["Optuna"]["model"],
               lgbm_models["LightGBM"]["GridSearch"]["model"],
               final_sb_model]

name_of_models = ["Decision Tree","Random Forest", "AdaBoost",
                  "Gradient Boosting", "CatBoost", "XGBoost",
                  "LightGBM", "SketchBoost"]

for name_of_model, model in zip(name_of_models, best_models):
    val_predict = model.predict(X_val)
    val_metrics = get_scores(y_val, val_predict)

    test_predict = model.predict(X_test)
    test_metrics = get_scores(y_test, test_predict)
    hold_kross_table.loc[name_of_model] = val_metrics + test_metrics

hold_kross_table

In [ ]:
sns.barplot(
    x='KFold R2',
    y=hold_kross_table.index,
    data=hold_kross_table.sort_values(by='KFold R2', ascending=True),
)

plt.title("R2 values")